# Licitacao Suspeita

Este notebook cruza **contratos governamentais** com **composicao societaria** de empresas e **doacoes eleitorais** para identificar padroes suspeitos de direcionamento de licitacoes.

**Fontes utilizadas:**
- CGU / Portal da Transparencia — contratos federais e licitacoes
- Receita Federal — QSA (socios das empresas contratadas)
- TSE — doacoes eleitorais (prestacao de contas)
- CGU — cadastros de sancoes (CEIS, CNEP)

**Campos-ponte:** CNPJ, CPF, Nome

**Dificuldade:** Avancado

---

## 1. Setup e configuracao

Instale as dependencias com `pip install -r requirements.txt` antes de executar.

In [ ]:
import pandas as pd
import requests
import time
import re
from pathlib import Path
from tabulate import tabulate

# ============================================================
# CONFIGURACAO
# ============================================================

# TODO: Insira sua chave de API do Portal da Transparencia
# Obtenha gratuitamente em: https://portaldatransparencia.gov.br/api-de-dados
API_KEY = "SEU_TOKEN_AQUI"

BASE_URL = "https://api.portaldatransparencia.gov.br/api-de-dados"
HEADERS = {"chave-api-dados": API_KEY, "Accept": "application/json"}

# TODO: Configure o periodo e orgao de interesse
CODIGO_ORGAO = ""  # Deixe vazio para todos os orgaos
DATA_INICIAL = "01/01/2024"
DATA_FINAL = "31/12/2024"

# Diretorio com dados da Receita Federal
# Download: https://dados.rfb.gov.br/CNPJ/
DIRETORIO_RFB = Path("./dados_rfb")

# Diretorio com dados do TSE
# Download: https://dadosabertos.tse.jus.br/
DIRETORIO_TSE = Path("./dados_tse")


def normalizar_cnpj(cnpj: str) -> str:
    """Remove formatacao do CNPJ e preenche com zeros."""
    if not cnpj:
        return ""
    return re.sub(r"[.\-/]", "", str(cnpj).strip()).zfill(14)


def coletar_paginado(endpoint: str, params: dict, max_paginas: int = 20) -> list:
    """Coleta dados paginados de um endpoint da API CGU."""
    todos = []
    for pagina in range(1, max_paginas + 1):
        try:
            params["pagina"] = pagina
            resp = requests.get(
                f"{BASE_URL}/{endpoint}",
                headers=HEADERS,
                params=params,
                timeout=30,
            )
            resp.raise_for_status()
            dados = resp.json()
            if not dados:
                break
            todos.extend(dados)
            time.sleep(2)  # Respeitar rate limit (30 req/min)
        except Exception as e:
            print(f"Erro no endpoint {endpoint}, pagina {pagina}: {e}")
            break
    return todos


print("Setup concluido.")
print(f"Periodo: {DATA_INICIAL} a {DATA_FINAL}")

## 2. Buscar contratos por orgao/periodo

Coletar todos os contratos federais vigentes no periodo especificado.

**Endpoint:** `GET https://api.portaldatransparencia.gov.br/api-de-dados/contratos`  
**Params:** `dataInicial`, `dataFinal`, `codigoOrgao` (opcional), `pagina`

In [ ]:
# TODO: Coletar contratos federais do periodo
# Endpoint: GET https://api.portaldatransparencia.gov.br/api-de-dados/contratos
# Params: dataInicial, dataFinal, codigoOrgao (opcional), pagina

# params_contratos = {
#     "dataInicial": DATA_INICIAL,
#     "dataFinal": DATA_FINAL,
# }
# if CODIGO_ORGAO:
#     params_contratos["codigoOrgao"] = CODIGO_ORGAO

# contratos_raw = coletar_paginado("contratos", params_contratos, max_paginas=50)
# df_contratos = pd.DataFrame(contratos_raw)

# TODO: Normalizar CNPJs e extrair campos relevantes
# if not df_contratos.empty:
#     df_contratos["cnpj_norm"] = df_contratos["cnpjContratado"].apply(normalizar_cnpj)
#     print(f"Contratos coletados: {len(df_contratos)}")
#     print(f"Empresas unicas contratadas: {df_contratos['cnpj_norm'].nunique()}")
#     print(f"Valor total: R$ {df_contratos['valorInicial'].sum():,.2f}")

print("TODO: Coletar contratos do periodo")

## 3. Extrair CNPJs das empresas contratadas

Criar lista unica de CNPJs das empresas que possuem contratos no periodo, preparando para consulta de socios.

In [ ]:
# TODO: Extrair lista unica de CNPJs contratados

# cnpjs_contratados = df_contratos["cnpj_norm"].unique().tolist()
# cnpjs_basicos = list(set(cnpj[:8] for cnpj in cnpjs_contratados))

# TODO: Resumir contratos por empresa
# resumo_contratos = df_contratos.groupby("cnpj_norm").agg(
#     num_contratos=("cnpjContratado", "count"),
#     valor_total=("valorInicial", "sum"),
#     orgaos=("nomeOrgao", lambda x: list(x.unique())),
# ).reset_index()

# print(f"CNPJs unicos (14 digitos): {len(cnpjs_contratados)}")
# print(f"CNPJs basicos (8 digitos): {len(cnpjs_basicos)}")

print("TODO: Extrair e resumir CNPJs contratados")

## 4. Consultar QSA (socios) de cada empresa

Para cada empresa contratada, buscar o **Quadro de Socios e Administradores (QSA)** na base da Receita Federal para identificar quem sao os donos.

**Fonte:** [Receita Federal — QSA](https://dados.rfb.gov.br/CNPJ/)  
**Arquivos:** `Socios0.csv` a `Socios9.csv`

In [ ]:
# TODO: Carregar QSA e buscar socios das empresas contratadas
# Download dos dados: https://dados.rfb.gov.br/CNPJ/dados_abertos_cnpj/
# Arquivos: Socios0.csv ... Socios9.csv (sem cabecalho)

COLUNAS_SOCIOS = [
    "cnpj_basico", "identificador_socio", "nome_socio_razao_social",
    "cpf_cnpj_socio", "qualificacao_socio", "data_entrada_sociedade",
    "pais", "representante_legal", "nome_representante",
    "qualificacao_representante_legal", "faixa_etaria",
]

# TODO: Carregar arquivos de socios
# dfs = []
# for i in range(10):
#     caminho = DIRETORIO_RFB / f"Socios{i}.csv"
#     if caminho.exists():
#         df = pd.read_csv(
#             caminho, sep=";", header=None, names=COLUNAS_SOCIOS,
#             dtype=str, encoding="latin-1",
#         )
#         dfs.append(df)
# df_socios = pd.concat(dfs, ignore_index=True)

# TODO: Filtrar socios das empresas contratadas
# socios_contratadas = df_socios[
#     df_socios["cnpj_basico"].isin(cnpjs_basicos)
# ].copy()
# socios_contratadas["nome_norm"] = (
#     socios_contratadas["nome_socio_razao_social"].str.upper().str.strip()
# )

# print(f"Socios encontrados: {len(socios_contratadas)}")
# print(f"Socios unicos: {socios_contratadas['nome_norm'].nunique()}")

print("TODO: Carregar QSA e filtrar socios das contratadas")

## 5. Cruzar socios com doacoes eleitorais (TSE)

Verificar se os socios das empresas contratadas fizeram **doacoes eleitorais** para candidatos. Isso pode indicar um esquema de "doacao -> contrato" (quid pro quo).

**Fonte:** [TSE — Prestacao de Contas](https://dadosabertos.tse.jus.br/)  
**Arquivo:** `receitas_candidatos_YYYY_BRASIL.csv`

In [ ]:
# TODO: Cruzar socios com doacoes eleitorais
# Download: https://dadosabertos.tse.jus.br/dataset/prestacao-de-contas-eleitorais
# Arquivo: receitas_candidatos_2022_BRASIL.csv

# Exemplo de carregamento:
# df_doacoes = pd.read_csv(
#     DIRETORIO_TSE / "receitas_candidatos_2022_BRASIL.csv",
#     sep=";",
#     encoding="latin-1",
#     dtype=str,
# )

# TODO: Normalizar nomes dos doadores
# df_doacoes["nome_doador_norm"] = df_doacoes["NM_DOADOR"].str.upper().str.strip()
# df_doacoes["cnpj_doador_norm"] = df_doacoes["NR_CPF_CNPJ_DOADOR"].apply(normalizar_cnpj)

# TODO: Cruzar por CNPJ da empresa (doacao de pessoa juridica)
# doacoes_empresas = df_doacoes[
#     df_doacoes["cnpj_doador_norm"].str[:8].isin(cnpjs_basicos)
# ]

# TODO: Cruzar por nome do socio (doacao de pessoa fisica)
# nomes_socios = set(socios_contratadas["nome_norm"].unique())
# doacoes_socios = df_doacoes[
#     df_doacoes["nome_doador_norm"].isin(nomes_socios)
# ]

# TODO: Consolidar achados
# Campos relevantes do TSE:
# - NM_DOADOR / NR_CPF_CNPJ_DOADOR (doador)
# - VR_RECEITA (valor doado)
# - NM_CANDIDATO (candidato beneficiado)
# - SG_PARTIDO (partido)
# - DS_CARGO (cargo do candidato)

print("TODO: Cruzar socios de contratadas com doacoes eleitorais")

## 6. Cruzar com sancoes (CEIS / CNEP)

Verificar se as empresas contratadas (ou seus socios) constam nos cadastros de sancoes.

**Endpoints:**
- CEIS: `GET https://api.portaldatransparencia.gov.br/api-de-dados/ceis`
- CNEP: `GET https://api.portaldatransparencia.gov.br/api-de-dados/cnep`

In [ ]:
# TODO: Verificar sancoes das empresas contratadas
# Endpoint CEIS: GET https://api.portaldatransparencia.gov.br/api-de-dados/ceis
#   Params: cnpjSancionado={CNPJ}, pagina
# Endpoint CNEP: GET https://api.portaldatransparencia.gov.br/api-de-dados/cnep
#   Params: cnpjSancionado={CNPJ}, pagina

# TODO: Coletar CEIS completo
# ceis_raw = coletar_paginado("ceis", params={}, max_paginas=100)
# df_ceis = pd.DataFrame(ceis_raw)
# if not df_ceis.empty:
#     df_ceis["cnpj_norm"] = df_ceis["cnpjSancionado"].apply(normalizar_cnpj)

# TODO: Coletar CNEP completo
# cnep_raw = coletar_paginado("cnep", params={}, max_paginas=50)
# df_cnep = pd.DataFrame(cnep_raw)
# if not df_cnep.empty:
#     df_cnep["cnpj_norm"] = df_cnep["cnpjSancionado"].apply(normalizar_cnpj)

# TODO: Cruzar CNPJs contratados com CEIS e CNEP
# cnpjs_set = set(cnpjs_contratados)
# alertas_ceis = cnpjs_set & set(df_ceis["cnpj_norm"].unique())
# alertas_cnep = cnpjs_set & set(df_cnep["cnpj_norm"].unique())

# if alertas_ceis:
#     print(f"[CRITICO] {len(alertas_ceis)} empresa(s) contratada(s) no CEIS!")
# if alertas_cnep:
#     print(f"[ALTO] {len(alertas_cnep)} empresa(s) contratada(s) no CNEP!")

print("TODO: Verificar sancoes das empresas contratadas")

## 7. Gerar relatorio de alertas

Consolidar todos os cruzamentos em um relatorio final, classificando por nivel de gravidade:

- **CRITICO:** empresa contratada esta no CEIS (impedida de licitar) E doou para campanhas
- **ALTO:** socio de empresa contratada doou para candidato eleito que controla o orgao contratante
- **MEDIO:** empresa contratada esta no CEIS ou CNEP (sem vinculo com doacoes)
- **BAIXO:** socio de empresa contratada fez doacoes eleitorais (sem sancao)

In [ ]:
# TODO: Gerar relatorio consolidado de alertas

# alertas = []

# TODO: Alertas de sancao + doacao (CRITICO)
# for cnpj in alertas_ceis:
#     if cnpj[:8] in [d[:8] for d in doacoes_empresas["cnpj_doador_norm"]]:
#         alertas.append({
#             "nivel": "CRITICO",
#             "cnpj": cnpj,
#             "motivo": "Empresa no CEIS com doacoes eleitorais e contratos ativos",
#             "valor_contratos": resumo_contratos[resumo_contratos["cnpj_norm"] == cnpj]["valor_total"].sum(),
#         })

# TODO: Alertas de doacao -> contrato (ALTO)
# TODO: Alertas de sancao sem doacao (MEDIO)
# TODO: Alertas de doacao sem sancao (BAIXO)

# TODO: Exibir relatorio
# df_alertas = pd.DataFrame(alertas)
# if not df_alertas.empty:
#     df_alertas = df_alertas.sort_values("nivel")
#     print("=" * 60)
#     print("RELATORIO DE LICITACOES SUSPEITAS")
#     print("=" * 60)
#     print(tabulate(df_alertas, headers="keys", tablefmt="grid"))
#     df_alertas.to_csv("relatorio_licitacoes_suspeitas.csv", index=False)
#     print(f"\nRelatorio salvo em: relatorio_licitacoes_suspeitas.csv")

print("TODO: Gerar relatorio consolidado de alertas")